# Ablation 11: Hard-Coulomb LSTM vs. TCN Backbone Tradeoff

This notebook isolates the final architectural question: once the Smooth Hard-Coulomb constraint makes unsafe discharge-direction trajectories unreachable, does the sequence backbone still matter?

The answer is deliberately split into two claims. The Hard-Coulomb output topology controls physical safety, so both backbones should report PVR = 0.00%. The LSTM/TCN backbone controls feature extraction inside that safe output manifold, so RMSE, MaxE, parameter count, and deployment cost can still differ.

## Scientific Motivation

An LSTM processes the window recurrently. It is compact and can in principle carry information across long histories, but inference is sequential and gradients can fade through time. A TCN uses causal dilated convolutions. It has a fixed receptive field, but it is parallelizable and often learns sharper transient features.

For SOC estimation, this distinction affects how well the network learns the hidden capacity map from voltage, current, temperature, and engineered features. It should not affect the PVR guarantee, because PVR is dictated by the Smooth Hard-Coulomb layer after the backbone.

In [1]:
from pathlib import Path
import json
import os

import numpy as np
import pandas as pd

if "JPY_PARENT_PID" not in os.environ:
    import matplotlib
    matplotlib.use("Agg")
import matplotlib.pyplot as plt

def find_repo_root(start=Path.cwd()):
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "outputs").exists() and (candidate / "JNTETI_Hard_Coulomb_LSTM_Draft.md").exists():
            return candidate
    raise FileNotFoundError("Could not locate repository root from current working directory.")

REPO_ROOT = find_repo_root()
FIG_DIR = REPO_ROOT / "outputs" / "figures" / "ablation_studies"
FIG_DIR.mkdir(parents=True, exist_ok=True)

SPRINT48_PATH = REPO_ROOT / "outputs" / "v7_final" / "sprint48_evaluation_results.json"
SPRINT52_PATH = REPO_ROOT / "outputs" / "v8_tcn_redemption" / "sprint52" / "sprint52_tcn_redemption_results.json"

print(f"Repository root: {REPO_ROOT}")
print(f"Figure output:   {FIG_DIR}")


Repository root: D:\Tugas kuliah\SEM 6\PROYEK DATA MINING\PENELITIAN\Penelitian_SOC
Figure output:   D:\Tugas kuliah\SEM 6\PROYEK DATA MINING\PENELITIAN\Penelitian_SOC\outputs\figures\ablation_studies


## Load Final Evaluation Ledgers

The comparison uses only the final post-refactor logs. Sprint 48 contains the clean Smooth Hard-Coulomb LSTM results. Sprint 52 contains the clean Smooth Hard-Coulomb TCN redemption results.

In [2]:
with SPRINT48_PATH.open(encoding="utf-8") as f:
    sprint48 = json.load(f)

with SPRINT52_PATH.open(encoding="utf-8") as f:
    sprint52 = json.load(f)

def one_match(records, predicate, label):
    matches = [record for record in records if predicate(record)]
    if len(matches) != 1:
        raise ValueError(f"Expected exactly one record for {label}, found {len(matches)}")
    return matches[0]

rows = []
for scenario in ["scenario_A", "scenario_B"]:
    lstm_record = one_match(
        sprint48,
        lambda r, s=scenario: r.get("model_kind") == "hard_coulomb_lstm" and r.get("scenario") == s,
        f"Hard-Coulomb LSTM {scenario}",
    )
    tcn_record = one_match(
        sprint52.get("results", []),
        lambda r, s=scenario: r.get("model") == "HardCoulombTCN" and r.get("scenario") == s,
        f"Hard-Coulomb TCN {scenario}",
    )

    for backbone, record, rf in [
        ("Hard-Coulomb LSTM", lstm_record, np.nan),
        ("Hard-Coulomb TCN", tcn_record, tcn_record.get("receptive_field", np.nan)),
    ]:
        metrics = record["metrics"]
        rows.append({
            "Scenario": scenario.replace("scenario_", "Scenario "),
            "Backbone": backbone,
            "RMSE (%)": float(metrics["rmse_full_pct"]),
            "MaxE (%)": float(metrics["maxe_full_pct"]),
            "PVR (%)": float(metrics["pvr_pct"]),
            "PVR violations": int(metrics.get("pvr_violations", 0)),
            "N windows": int(metrics["n_windows"]),
            "Parameters": int(record["parameter_count"]),
            "Receptive field": rf,
        })

df = pd.DataFrame(rows)
lstm_params = int(df.loc[df["Backbone"] == "Hard-Coulomb LSTM", "Parameters"].iloc[0])
df["Parameter ratio vs LSTM"] = df["Parameters"].map(lambda p: f"{p / lstm_params:.2f}x")

display_df = df.copy()
for col in ["RMSE (%)", "MaxE (%)", "PVR (%)"]:
    display_df[col] = display_df[col].map(lambda x: f"{x:.2f}")
display_df["Parameters"] = display_df["Parameters"].map(lambda x: f"{x:,}")
display_df["Receptive field"] = display_df["Receptive field"].map(lambda x: "-" if pd.isna(x) else f"{int(x)} steps")

try:
    display(display_df)
except NameError:
    print(display_df.to_string(index=False))

assert np.allclose(df["PVR (%)"].to_numpy(), 0.0), "PVR must remain 0.00% for both backbones."
assert int(df["PVR violations"].sum()) == 0, "PVR violation count must be exactly zero."
print("PVR certification passed: Hard-Coulomb LSTM and TCN both report 0.00% PVR with zero violation counts.")


,Scenario,Backbone,RMSE (%),MaxE (%),PVR (%),PVR violations,N windows,Parameters,Receptive field,Parameter ratio vs LSTM
0,Scenario A,Hard-Coulomb LSTM,12.71,55.11,0.00,0,18914,"54,626",-,1.00x
1,Scenario A,Hard-Coulomb TCN,11.46,46.73,0.00,0,18914,"208,546",181 steps,3.82x
2,Scenario B,Hard-Coulomb LSTM,8.57,35.00,0.00,0,8010,"54,626",-,1.00x
3,Scenario B,Hard-Coulomb TCN,8.58,39.49,0.00,0,8010,"208,546",181 steps,3.82x


PVR certification passed: Hard-Coulomb LSTM and TCN both report 0.00% PVR with zero violation counts.


## Comparative Backbone Figure

The grouped bars separate accuracy from safety. RMSE and MaxE move with the backbone, but PVR remains pinned at zero because the final layer constrains the reachable output set.

In [3]:
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "DejaVu Serif"],
    "font.size": 9.5,
    "axes.labelsize": 9.5,
    "axes.titlesize": 10.5,
    "axes.titleweight": "bold",
    "legend.fontsize": 8.5,
    "legend.frameon": False,
    "figure.dpi": 300,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.18,
    "grid.linestyle": "-",
})

scenarios = ["Scenario A", "Scenario B"]
backbones = ["Hard-Coulomb LSTM", "Hard-Coulomb TCN"]
colors = {"Hard-Coulomb LSTM": "#264653", "Hard-Coulomb TCN": "#E76F51"}
x = np.arange(len(scenarios))
bar_width = 0.36

fig, axes = plt.subplots(1, 3, figsize=(7.25, 2.85), gridspec_kw={"width_ratios": [1.0, 1.0, 0.85]})

def values(metric, backbone):
    return [
        float(df.loc[(df["Scenario"] == scenario) & (df["Backbone"] == backbone), metric].iloc[0])
        for scenario in scenarios
    ]

def annotate(ax, bars, fmt="{:.2f}", pad=0.35, fontsize=7.6):
    for bar in bars:
        height = bar.get_height()
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            height + pad,
            fmt.format(height),
            ha="center",
            va="bottom",
            fontsize=fontsize,
            color="#333333",
        )

for axis, metric, ylabel, ylim, pad in [
    (axes[0], "RMSE (%)", "RMSE (%)", 15.0, 0.25),
    (axes[1], "MaxE (%)", "MaxE (%)", 60.0, 0.9),
    (axes[2], "PVR (%)", "PVR (%)", 1.0, 0.035),
]:
    for i, backbone in enumerate(backbones):
        offset = (i - 0.5) * bar_width
        metric_values = values(metric, backbone)
        bars = axis.bar(
            x + offset,
            metric_values,
            bar_width * 0.92,
            label=backbone,
            color=colors[backbone],
            edgecolor="white",
            linewidth=0.6,
            zorder=3,
        )
        annotate(axis, bars, pad=pad, fontsize=7.4 if metric != "PVR (%)" else 7.8)
    axis.set_xticks(x)
    axis.set_xticklabels(["A\nOOD", "B\nChron."])
    axis.set_ylabel(ylabel)
    axis.set_ylim(0, ylim)
    axis.set_axisbelow(True)

axes[0].set_title("Average error")
axes[1].set_title("Worst-case error")
axes[2].set_title("Safety invariant")
axes[2].text(0.5, 0.78, "0 violations", transform=axes[2].transAxes, ha="center", va="center", fontsize=8.5, fontweight="bold", color="#264653")

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=2, bbox_to_anchor=(0.5, 1.08))

tcn_params = int(df.loc[df["Backbone"] == "Hard-Coulomb TCN", "Parameters"].iloc[0])
tcn_rf = int(df.loc[df["Backbone"] == "Hard-Coulomb TCN", "Receptive field"].dropna().iloc[0])
fig.text(
    0.5,
    -0.08,
    f"TCN uses {tcn_params:,} parameters ({tcn_params / lstm_params:.2f}x LSTM) with a {tcn_rf}-step receptive field; both backbones preserve PVR = 0.00%.",
    ha="center",
    fontsize=8.2,
    color="#333333",
)

fig.tight_layout(w_pad=1.15)
pdf_path = FIG_DIR / "fig_11_hardcoulomb_lstm_vs_tcn_tradeoff.pdf"
png_path = FIG_DIR / "fig_11_hardcoulomb_lstm_vs_tcn_tradeoff.png"
fig.savefig(pdf_path)
fig.savefig(png_path, dpi=300)
plt.show()

print(f"Saved PDF: {pdf_path}")
print(f"Saved PNG: {png_path}")
assert pdf_path.exists() and pdf_path.stat().st_size > 0
assert png_path.exists() and png_path.stat().st_size > 0


Saved PDF: D:\Tugas kuliah\SEM 6\PROYEK DATA MINING\PENELITIAN\Penelitian_SOC\outputs\figures\ablation_studies\fig_11_hardcoulomb_lstm_vs_tcn_tradeoff.pdf
Saved PNG: D:\Tugas kuliah\SEM 6\PROYEK DATA MINING\PENELITIAN\Penelitian_SOC\outputs\figures\ablation_studies\fig_11_hardcoulomb_lstm_vs_tcn_tradeoff.png


C:\Users\VICTUS\AppData\Local\Temp\ipykernel_37592\1009811085.py:96: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Interpretation

The TCN is stronger on Scenario A, the harder out-of-distribution split: it reduces RMSE from 12.71% to 11.46% and MaxE from 55.11% to 46.73%. On Scenario B, the LSTM has nearly identical RMSE and lower MaxE than the TCN. The TCN pays for its Scenario A gain with a larger model: 208,546 parameters versus 54,626 for the LSTM, or about 3.82x.

The scientific conclusion is therefore not that one backbone is universally superior. The conclusion is that Hard-Coulomb safety is backbone-agnostic, while accuracy and embedded deployment cost remain backbone-dependent.